In [ ]:
%load_ext autoreload
%autoreload 2
import waffles
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

from waffles.data_classes.Waveform import Waveform
from waffles.data_classes.WaveformSet import WaveformSet
from waffles.data_classes.UniqueChannel import UniqueChannel
from waffles.data_classes.ChannelWsGrid import ChannelWsGrid
from waffles.utils.numerical_utils import average_wf_ch
from waffles.utils.selector_waveforms import WaveformSelector
from waffles.np02_utils.AutoMap import dict_uniqch_to_module, dict_module_to_uniqch, strUch, ordered_channels_membrane, ordered_channels_cathode
from waffles.np02_utils.AutoMap import  ordered_modules_pmt, ordered_modules_cathode, ordered_modules_membrane, getModuleName, getEndpointChannelFromModule
from waffles.np02_utils.PlotUtils import plot_detectors, runBasicWfAnaNP02
from waffles.np02_utils.PlotUtils import matplotlib_plot_WaveformSetGrid, wfset_remove_bad_baselines
from waffles.np02_utils.load_utils import open_processed, remove_extra_channels_membrane

import mplhep
mplhep.style.use(mplhep.style.ROOT)
plt.rcParams.update({'font.size': 12,
                     'grid.linestyle': '--',
                     'axes.grid': True,
                     'figure.autolayout': True,
                     'figure.figsize': [14,6]
                     })

In [ ]:
run = 43288
dettype = "membrane"
# dettype = "cathode"
# dettype = "pmt"
nwaveforms = 100000

datadir = f"/eos/experiment/neutplatform/protodune/experiments/ProtoDUNE-VD/commissioning/"
endpoint = 106 if dettype == "cathode" else 107 if dettype== "membrane" else 110
listmods = ordered_modules_cathode if endpoint == 106 else ordered_modules_membrane if endpoint == 107 else ordered_modules_pmt

n = len(listmods)
group1 = listmods[:n//4]
group2 = listmods[n//4:n//2]
group3 = listmods[n//2:2*(n//3+1)]
group4 = listmods[2*(n//3+1):]

groupLow = group1+group2
groupHig = group3+group4
groupall = group1+group2+group3+group4

In [ ]:
wfset_full = open_processed(run, dettype, datadir, nwaveforms=nwaveforms, mergefiles=True, file_slice=slice(None,None))
# wfset_cathode = open_processed(run, "cathode", datadir, nwaveforms=nwaveforms, mergefiles=True, file_slice=slice(None,None))
# wfset_full.merge(wfset_cathode)
wfset_full = WaveformSet.from_filtered_WaveformSet(wfset_full, remove_extra_channels_membrane)

In [ ]:
runBasicWfAnaNP02(wfset_full, onlyoptimal=False, baselinefinish=60, int_ll=64, int_ul=400, amp_ll=64, amp_ul=150, configyaml="")

In [ ]:
wfset_full = wfset_remove_bad_baselines(wfset_full)
wfset_full

In [ ]:
argsheat = dict(
    mode="heatmap",
    analysis_label="std",
    adc_range_above_baseline=8000,
    adc_range_below_baseline=-1250,
    adc_bins=250,
    time_bins=wfset_full.points_per_wf//2,
    filtering=8,
    share_y_scale=False,
    share_x_scale=True,
    wfs_per_axes=5000,
    heatmap_chunk_size = 5000,
    zlog=True,
    width=1300,
    height=650*4,
    showplots=True,
    cols=2,
)

#detector = groupLow
# detector = groupHig
detector = groupall

#detector=["M7(1)","M7(2)"]
#detector=["C1(1)"]

plot_detectors(wfset_full, detector, **argsheat)

In [ ]:
extractor = WaveformSelector('cuts.yaml')

In [ ]:
extractor.loadcuts()
wfset_clean = WaveformSet.from_filtered_WaveformSet(wfset_full, extractor.applycuts, show_progress=True)
print(f"Original waveforms: {len(wfset_full.waveforms)}, after cut: {len(wfset_clean.waveforms)}")

In [ ]:
argsheat['adc_range_above_baseline']=8000
argsheat['adc_range_below_baseline']=-2250
argsheat['adc_bins']=400
argsheat['wfs_per_axes']=500e3
argsheat['share_y_scale']=True
argsheat['filtering'] = 32
plot_detectors(wfset_clean, detector, **argsheat)

In [ ]:
wfsetch = ChannelWsGrid.clusterize_waveform_set(wfset_clean)#[endpoint]

In [ ]:
test = {}
for wf in wfset_full.waveforms:
    test[wf.daq_window_timestamp] =  test.get(wf.daq_window_timestamp, []) + [wf.record_number]
    
test = { k: set(v) for k, v in test.items() } 

for t in test.values():
    if len(t) > 1:
        raise Exception("Wtf!?")

In [ ]:
def filter_selected(wf: Waveform, ts_selected:dict):
    if wf.record_number in ts_selected:
        if wf.timestamp in ts_selected[wf.record_number]:
            return True
    return False
def checkchannel(list_idx, wfset):
    for idx in list_idx:
        wf = wfset.waveforms[idx]
        mname = getModuleName(wf.endpoint, wf.channel)
        if mname == "M7(1)":
            return True
    return True
def create_matching_events(module, wfsetch, wfset_full):
    ep, ch = getEndpointChannelFromModule(module)
    records_timestamps_selected = {}
    for wf in wfsetch[ep][ch].waveforms:
        records_timestamps_selected[wf.record_number] = records_timestamps_selected.get(wf.record_number, []) + [wf.timestamp]
    
    for k, v in records_timestamps_selected.items():
        records_timestamps_selected[k] = np.array(v)
    
    toplot_idx = {}
    toplot_timestamps = {}
    for i, wf in enumerate(tqdm(wfset_full.waveforms)):
        timestamp = wf.timestamp
        record_number = wf.record_number
        # Max allowed = 1024
        delta_ticks = 200
        if record_number in records_timestamps_selected:
            diff = np.abs(records_timestamps_selected[record_number] - timestamp)
            match = np.where(diff <= delta_ticks)[0]
            if len(match) > 0:
                if len(match) > 1:
                    raise Exception("This should not happen...")
                timestampmatch = records_timestamps_selected[record_number][match[0]]
                toplot_idx[timestampmatch] = toplot_idx.get(timestampmatch, []) + [i]
                # toplot_timestamps[record_number] = toplot_timestamps.get(record_number, []) + [timestamp]
                
    toplot_idx = { k: v for k, v in sorted(toplot_idx.items(), key=lambda x: x[0]) if checkchannel(v, wfset_full) }
    list_all_indexes = [ v for l in toplot_idx.values() for v in l ]

    print(len(list_all_indexes))
    for idx in list_all_indexes:
        wf = wfset_full.waveforms[idx]
        record_number = wf.record_number
        timestamp = wf.timestamp
        toplot_timestamps[record_number] = toplot_timestamps.get(record_number, []) + [timestamp]
    toplot_timestamps = { k: v for k, v in sorted(toplot_timestamps.items(), key=lambda x: x[0]) }
    
    wfset_selected = WaveformSet.from_filtered_WaveformSet(wfset_full, filter_selected, toplot_timestamps, show_progress=True)
    print('n records:', len(toplot_idx))
    print('total timestamps:', len( [ v for lts in list(toplot_idx.values()) for v in lts ] ))
    return toplot_idx, wfset_selected

In [ ]:
module = "M8(1)"
toplot_idx, wfset_selected = create_matching_events(module, wfsetch, wfset_full)
wfset_selected

In [ ]:
from waffles.np02_utils.load_utils import ch_read_template
from utils_conv import ConvFitterVDWrapper, process_convfit
templates = ch_read_template('templates_even_larger')
def plot_selected(wfsetch:WaveformSet, wfset_full:WaveformSet,  toplot:dict, evt = 0, several=False):
    if several:
        evt = min(evt, len(toplot.keys()))
        evt = range(evt)
    else:
        evt = [evt]
    counts = 0
    for e in evt:
        reference_timestamp = list(toplot.keys())[e]
        indexes_wfs = toplot[reference_timestamp]
        wfm = wfsetch.waveforms[0]
        ep, ch = wfm.endpoint, wfm.channel
        if ep in templates and ch in templates[ep]:
            tp = templates[ep][ch]
        else:
            continue
        for idx_wf in indexes_wfs:
            wf:Waveform = wfset_full.waveforms[idx_wf]
            if ep != wf.endpoint or ch != wf.channel:
                continue
            offset = wf.timestamp - reference_timestamp
            x = np.linspace(-offset, wfset_full.points_per_wf - offset, wfset_full.points_per_wf, endpoint=False)
            plt.plot(x, wf.adcs - wf.analyses['std'].result['baseline'])
            counts += 1
            # cfit = ConvFitterVDWrapper(usemplhep=False)
            # process_convfit(ep, ch, response=wf.adcs - wf.analyses['std'].result['baseline'], template=tp, cfitch = cfit)
            # cfit.plot(newplot=False)
            # plt.legend(fontsize=8)
            
    # # plt.xlim(40, 80)
    plt.xlabel('Relative time [ticks]')
    plt.ylabel('Amplitude [ADC]')
    title = getModuleName(ep, ch)
    if not several:
        title += f' - evt: {evt}'
    else:
        title += f' - {counts} events'
    plt.title(title)
    

In [ ]:
# from waffles.np02_utils.AutoMap import generate_ChannelMap
# from waffles.data_classes.ChannelWsGrid import ChannelWsGrid
# from waffles.np02_utils.PlotUtils import matplotlib_plot_ChannelWsGrid
# from PIL import Image

# images = []
# detChMap = generate_ChannelMap(channels=['M2', 'M4', 'M5', 'M7', 'M6', 'M8'], cols = 4)
# gridWs = ChannelWsGrid(detChMap, wfset_selected)
# for evt in tqdm(range(100)):
#     func_params = dict(
#         wfset_full = wfset_full,
#         toplot = toplot_idx,
#         evt = evt,
#         several = False
#     )
#     fig, ax = matplotlib_plot_ChannelWsGrid(gridWs, plot_function=plot_selected, func_params=func_params)
#     fig.suptitle(f"Matching selected {module}")
#     fig.subplots_adjust(top=0.88)
#     # plt.tight_layout()
#     plt.savefig('tmp.png')
#     images.append(Image.open('tmp.png'))
#     images[0].save('animated_plot.gif', save_all=True, append_images=images, duration=300, loop=0)
#     plt.close()


In [ ]:
evt = 100
func_params = dict(
    wfset_full = wfset_full,
    toplot = toplot_idx,
    evt = evt,
    several = True
)
fig, ax = matplotlib_plot_WaveformSetGrid(wfset_selected, detector=['M2', 'M4', 'M5', 'M7', 'M6', 'M8'], plot_function=plot_selected, func_params=func_params, figsize=(22,8), cols=4)
fig.suptitle(f"Matching selected {module}")
fig.subplots_adjust(top=0.88)
plt.tight_layout()
# plt.savefig(f'./graphs_check/cumulative_{module}.png')
fig, ax = matplotlib_plot_WaveformSetGrid(wfset_selected, detector=['C'], plot_function=plot_selected, func_params=func_params, figsize=(22,8), cols=4)
fig.suptitle(f"Matching selected {module}")
fig.subplots_adjust(top=0.88)
plt.tight_layout()

In [ ]:

argsheat['adc_range_above_baseline']=8000
argsheat['adc_range_below_baseline']=-2250
argsheat['adc_bins']=400
argsheat['wfs_per_axes']=None
argsheat['share_y_scale']=True
argsheat['filtering'] = 0
argsheat['cols'] = 4
argsheat['width'] = 1600
argsheat['height'] = 800
plot_detectors(wfset_selected,  detector=['M2', 'M4', 'M5', 'M7', 'M6', 'M8'], **argsheat)

In [ ]:
from waffles.np02_utils.load_utils import ch_read_calib
dcalib = ch_read_calib()

In [ ]:
def plot_pe(wfset:Waveform):
    ep = wfset.waveforms[0].endpoint
    ch = wfset.waveforms[0].channel
    charges = np.array([ wf.analyses['std'].result['integral'] for wf in wfset.waveforms ] )
    blow, bhig = np.quantile(charges, [0.02, 0.98])
    charges = charges[ (charges > blow) & (charges < bhig) ]
    pes = charges / dcalib[ep][ch]['Gain']
    mean = np.mean(pes)
    plt.hist(pes, bins=50)
    plt.axvline(mean, ls='--', c='r')
    plt.xlabel('Photo-electrons')
    plt.title(label=getModuleName(ep, ch))
    

In [ ]:
fig, ax = matplotlib_plot_WaveformSetGrid(wfset_selected, detector=['M2', 'M4', 'M5', 'M7', 'M6', 'M8'], plot_function=plot_pe, cols=4)
# plt.savefig(f'./graphs_check/PEs_{module}.png')

In [ ]:

fig, ax = matplotlib_plot_WaveformSetGrid(wfset_selected, detector=['C'], plot_function=plot_pe, cols=4)